In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os
current_user = spark.sql("select current_user()").collect()[0]['current_user()']
os.chdir(f"/Workspace/Users/{current_user}/chaosllama")

In [0]:
%pip install -r requirements.txt
%pip install mlflow==3.6.0
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os
current_user = spark.sql("select current_user()").collect()[0]['current_user()']
os.chdir(f"/Workspace/Users/{current_user}/chaosllama")

In [0]:
CATALOG = "data_science_dev"
SCHEMA = "ask_contact_center"
EVAL_TABLE_NAME = "chaos_llama_queries_v2"

In [0]:
eval_dataset = spark.table(f"{CATALOG}.{SCHEMA}.{EVAL_TABLE_NAME}")
eval_dataset.display()

In [0]:
%sql

DROP TABLE IF EXISTS data_science_dev.ask_contact_center.gold_chaos_llama_evalset;


In [0]:
import pandas as pd
from mlflow.genai import datasets
from pyspark.sql.types import StructType, StructField, MapType, StringType 

# Use apply to build the new structure

def create_eval_dataset(eval_dataset:pd.DataFrame):
    def to_row(row):
        return pd.Series(
            {
                "inputs": {"question": row["question"], "timeout": row["timeout"]},
                "expectations": {"expected_response": row["ground_truth_sql"]},
            }
        )

    schema = StructType([
        StructField("inputs", MapType(StringType(), StringType()), True),
        StructField("expectations", MapType(StringType(), StringType()), True)
    ])
    eval_dataset["timeout"] = [ str(i * 12) for i in range(len(eval_dataset))]
    new_df = eval_dataset.apply(to_row, axis=1)
    spark.createDataFrame(new_df, schema=schema).write.option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA}.gold_chaos_llama_evalset", mode="overwrite")
    spark.table(f"{CATALOG}.{SCHEMA}.gold_chaos_llama_evalset").display()

def create_or_get_mlflow_eval_dataset(eval_dataset):
    try:
        mlflow_eval_dataset = datasets.create_dataset(eval_dataset)
        return mlflow_eval_dataset
    except Exception as e:
        print("Dataset already exists, getting existing one....")
        mlflow_eval_dataset = datasets.get_dataset(eval_dataset)
        return mlflow_eval_dataset


create_eval_dataset(eval_dataset.toPandas())


In [0]:
from chaosllama.services import judges
from chaosllama.scorers.scorers import *
from rich.pretty import pprint

def get_judges(filter_cond:list[str]=None):
    jmngr = judges.JudgeService(scorers=[eval_sql_clauses_distro,eval_query_results]) 
    jmngr.load_judges_from_config()
    judge_list = jmngr.judges + jmngr.scorers
    if filter_cond:
        judge_list = [j for j in judge_list if j.name in filter_cond ]
        return judge_list
    else:
        return judge_list

pprint(get_judges())

In [0]:
import mlflow

CHAOS_LLAMA_PROMPT_REGISTRY = f"{CATALOG}.{SCHEMA}.chaosllama_prompt_registry"
GENIE_TEMPLATE = """{{system_instructions}}"""
GENIE_TEMPLATE_V2 = """Follow these guidelines before answering the question"""

def register_prompt():
    genie_prompt = mlflow.genai.register_prompt(name=f"{CHAOS_LLAMA_PROMPT_REGISTRY}", template=GENIE_TEMPLATE)
    return genie_prompt

def get_prompt(version=1):
    return mlflow.genai.load_prompt(name_or_uri=f"{CHAOS_LLAMA_PROMPT_REGISTRY}", version=version)



#prompt=register_prompt
#get_prompt(version=prompt.version)

In [0]:
import warnings
import logging
def suppress_warnings(modules=["mlflow.genai.judges.instructions_judge"]):  
    for module in modules:
        logging.getLogger("mlflow.genai.judges.instructions_judge").setLevel(logging.ERROR)    

In [0]:
from mlflow.genai.optimize import GepaPromptOptimizer
from chaosllama.prompts.registry import INSTROSPECT_PROMPT_V6
from mlflow.genai import datasets
from chaosllama.services.genie import GenieAgent_v2
from chaosllama.profiles.config import config
import mlflow
from functools import partial
from pprint import pprint
# from chaosllama

GENIE_TEMPLATE = "{{system_instructions}}"
INSTROPECTION_TEMPLATE = INSTROSPECT_PROMPT_V6
GOLD_EVALSET = f"{CATALOG}.{SCHEMA}.gold_chaos_llama_evalset"
REFLECTION_LLM = "databricks:/databricks-claude-sonnet-4-5"
JUDGE_LLM = "databricks:/databricks-claude-3-7-sonnet"

mlflow_eval_dataset = spark.table(GOLD_EVALSET).toPandas()
mlflow_eval_dataset = mlflow_eval_dataset[
    mlflow_eval_dataset["inputs"].apply(lambda d: d.get("question") != "What is the total amount of money spent on retail items for each contract for customer '044acb7e-249a-11e4-a0e7-005056811110'?")
].iloc[0:5]
display(mlflow_eval_dataset)
mlflow.set_experiment("/Users/akil.thomas@databricks.com/GEPA OPTIMIZER")
suppress_warnings()

#aggregations=lambda values: sum([value.lower() == 'true' for value in values]
def objective_fn(scorers_map, weight_map=False):
    aggregations = {}
    for k, feedback in scorers_map.items():
        if feedback.name == "sql_results_equivalence": print(f"💭 {feedback.metadata["inputs"]["question"]}")
        if isinstance(feedback, Feedback):
            if isinstance(feedback.value, str):
                converted_val = float(feedback.value.lower() == "true")
            elif isinstance(feedback.value, bool):
                converted_val = float(feedback.value)
            aggregations[k] = converted_val
        else:
            raise NotImplemntedError

    
    print(f"\t🧮{aggregations=}")
    if weight_map:
        WEIGHT_MAP = {
            'sql_semantic_equivalence': .10, 
            #'sql_clauses_distribution_equivalence': .10, 
            'sql_results_equivalence': .90
        }
        score = sum([ WEIGHT_MAP[k] * v for k,v in aggregations.items() ])
        print(f"\t🥅{score=}")
        return score
    else:
        return sum(aggregations.values())

objective_fn_partial = partial(objective_fn, weight_map=True)

result = mlflow.genai.optimize_prompts(
    predict_fn=GenieAgent_v2().invoke,
    train_data=mlflow_eval_dataset,
    prompt_uris=[f"prompts:/{config.runtime.PROMPT_REGISTRY}/1"],
    optimizer=GepaPromptOptimizer(
        reflection_model=REFLECTION_LLM, 
        max_metric_calls=100,
        display_progress_bar=True,
    ),
    scorers=get_judges(["sql_results_equivalence", "sql_semantic_equivalence"]),
    enable_tracking=True,
    aggregation=objective_fn_partial,
)

In [0]:
result

In [0]:
from chaosllama.introspection.gepa_introspection import test_dspy_introspection

optimized_prompt = test_dspy_introspection()


In [0]:
from dspy.datasets.dataset import Dataset as DspyDataset
from sklearn.model_selection import train_test_split

class DspyEvalSet(DspyDataset):
    def __init__(
        self,*args, test_perc:float=0.30,**kwargs,
    ) -> None:

        super().__init__(*args, **kwargs)
        self.test_perc = test_perc
        self._create_train_test_split_and_ensure_labels()

    def _create_train_test_split_and_ensure_labels(self) -> None:
        """Perform a train/test split that ensure labels in `test` are also in `train`."""
        # Read the data
        data = get_eval_set() #self.eval_mngr.eval_set.data
        train_df, test_df = train_test_split(data, test_size=self.test_perc, random_state=1)

        # Set DSPy class variables
        self._train = train_df.to_dict(orient="records")
        self._test = test_df.to_dict(orient="records")

def get_eval_set():
    return spark.table(f"{config.CATALOG}.{config.SCHEMA}.chaos_llama_queries").toPandas()

data = DspyEvalSet()

data.train